***

##  MLOPS - PRACTICAL 4

***

## aim:

To use Git for version control of an MLOps project and implement model training, validation, saving and inference through a Python application.

## problem statement:

Create a Git repository for a customer-churn MLOps project, commit and push the project to GitHub. Train a Random Forest model, validate it against an accuracy threshold, save it, and provide prediction through a Flask API.

***

## git commands:

In [ ]:
git init

git config --global user.name "Your Name"
git config --global user.email "your_email@gmail.com"

git status

git add .

git commit -m "Initial commit"

git log
git log --oneline

git remote add origin https://github.com/yourusername/Customer-Churn-Predictor.git

git remote -v

git branch -M main

git push -u origin main

### git workflow:

**Working Directory → `git add` → Staging Area → `git commit` → Local Repository → `git push` → Remote Repository**

***

## python code:

### 1. training and validation - `train.py`

In [2]:
import sys
import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

ACCURACY_THRESHOLD = 0.80

def load_data():
    X, y = make_classification( n_samples=2000, n_features=12, n_informative=8,
                               weights=[0.7, 0.3],  random_state=42 )
    return X, y


def main():

    X, y = load_data()

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y )

    model = RandomForestClassifier(n_estimators=200,  max_depth=8, random_state=42)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    print(f"Validation Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds))

    joblib.dump(model, "model.joblib")
    print("Model saved to model.joblib")

    if acc < ACCURACY_THRESHOLD:
        print(f"FAILED: accuracy {acc:.4f} below threshold {ACCURACY_THRESHOLD}")
        sys.exit(1)

    print("Model validation PASSED")

if __name__ == "__main__":
    main()

Validation Accuracy: 0.8850
              precision    recall  f1-score   support

           0       0.89      0.95      0.92       279
           1       0.86      0.74      0.79       121

    accuracy                           0.89       400
   macro avg       0.88      0.84      0.86       400
weighted avg       0.88      0.89      0.88       400

Model saved to model.joblib
Model validation PASSED


### 2. inference api - `app.py`

In [4]:
from flask import Flask, request, jsonify
import joblib
import numpy as np

# Create Flask application
app = Flask(__name__)

# Load trained model
model = joblib.load("model.joblib")


# Home route
@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Customer Churn Prediction API is running"
    }), 200


# Health-check route
@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "healthy"
    }), 200


# Prediction route
@app.route("/predict", methods=["POST"])
def predict():

    # Get JSON input
    data = request.get_json()

    # Extract features
    features = np.array(
        data["features"]
    ).reshape(1, -1)

    # Make prediction
    prediction = model.predict(features)[0]

    # Get prediction probabilities
    probability = model.predict_proba(
        features
    )[0].tolist()

    # Return result as JSON
    return jsonify({
        "prediction": int(prediction),
        "churn_probability": probability[1]
    }), 200


# Run Flask application
if __name__ == "__main__":
    app.run(
        host="0.0.0.0",
        port=8080,
        debug=False
    )

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8080
 * Running on http://192.168.29.85:8080
Press CTRL+C to quit
192.168.29.85 - - [30/Aug/2026 22:31:13] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [30/Aug/2026 22:31:15] "GET / HTTP/1.1" 404 -


***

## supporting files:

### `requirements.txt`

Contains the Python packages required by the application.

### `buildspec.yml`

Defines the CodeBuild stages and commands. The important logic is:

```yaml
phases:
  install:
    runtime-versions:
      python: 3.11
    commands:
      - pip install -r requirements.txt

  build:
    commands:
      - python train.py
      - mkdir -p build_output
      - cp model.joblib app.py appspec.yml requirements.txt build_output/
      - cp -r scripts build_output/
```

If `train.py` exits with a non-zero status because accuracy is below the threshold, the build stage fails.

### `appspec.yml`

Defines CodeDeploy deployment instructions:

```yaml
version: 0.0
os: linux

files:
  - source: /
    destination: /home/ec2-user/churn-app

hooks:
  BeforeInstall:
    - location: scripts/install_dependencies.sh
      timeout: 300
      runas: root

  ApplicationStart:
    - location: scripts/start_server.sh
      timeout: 120
      runas: root

  ApplicationStop:
    - location: scripts/stop_server.sh
      timeout: 60
      runas: root
```

***

## output:

The training program prints validation accuracy and a classification report, saves `model.joblib`, and prints either:

Model validation PASSED or FAILED: accuracy ... below threshold 0.8

The Flask application provides:
- GET  /health 
- POST /predict


## viva questions:

1. **What does `git init` do?**  
   Initializes a Git repository.

2. **What does `git add` do?**  
   Moves changes to the staging area.

3. **What does `git commit` do?**  
   Creates a snapshot of staged changes in the local repository.

4. **What does `git push` do?**  
   Sends local commits to the remote repository.

5. **Difference between commit and push?**  
   Commit saves locally; push sends the commit to the remote repository.

6. **What is GitHub?**  
   A platform for hosting Git repositories and collaboration.

7. **Why use an accuracy threshold?**  
   To prevent a model with unacceptable validation performance from progressing.

8. **What does `sys.exit(1)` indicate?**  
   Failure/non-successful execution.

9. **What does `sys.exit(0)` indicate?**  
   Successful execution.

10. **Why use `/health`?**  
    To check whether the service is running.

11. **What does `request.get_json()` do?**  
    Reads JSON data sent to the API.

12. **What does `predict_proba()` return?**  
    Class probabilities.

13. **Why use `reshape(1, -1)`?**  
    To make one input observation into the 2D shape expected by the model.

14. **What is CI/CD?**  
    Automation of software integration, testing, delivery and/or deployment stages.

***

## result:

Git provides version control for the MLOps project, while the Python workflow trains and validates the model, saves the approved model, and exposes inference through a Flask API.

## observation:

The MLOps project was version-controlled using Git. The Python workflow trains and validates the model, saves the model only as part of the successful workflow, and the Flask application provides health-check and prediction endpoints. Git enables versioning and collaboration, while the validation threshold helps prevent a low-performing model from progressing.

***